# NLP Lab 1 — Part 2: Text Classification

Dataset: [avsolatorio/mteb-emotion-avs_triplets](https://huggingface.co/datasets/avsolatorio/mteb-emotion-avs_triplets).

Dependencies: `numpy`, `datasets`, and `scikit-learn`.

In [ ]:
import re
from time import perf_counter

import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


CLASSES = ["sadness", "joy", "love", "anger", "fear", "surprise"]

## Preprocessing

Text was converted to lowercase, punctuation and numbers were replaced with
spaces, and stopwords were retained. Each vocabulary was built from the
training set only. Unseen test features were ignored.

In [2]:
def preprocess(text):
    # Lowercase, replace punctuation/numbers with spaces, keep stopwords.
    return " ".join(re.sub(r"[^a-z]+", " ", text.lower()).split())


def get_bigrams(text):
    words = text.split()
    return [words[i] + " " + words[i + 1] for i in range(len(words) - 1)]


def make_vocabulary(documents):
    tokens = sorted({token for document in documents for token in document})
    return {token: i for i, token in enumerate(tokens)}


def encode(documents, vocabulary, binary=False):
    # Store only known token IDs. For binary features, each ID appears once.
    result = []
    for document in documents:
        ids = [vocabulary[token] for token in document if token in vocabulary]
        if binary:
            ids = sorted(set(ids))
        result.append(np.array(ids, dtype=int))
    return result

A stratified 80/20 train/test split was used with seed 42 for all models.
The dataset was retained without changes. Thirty repeated sentences with
conflicting labels are included; repeated sentences may occur in both splits.

In [3]:
data = load_dataset("avsolatorio/mteb-emotion-avs_triplets", split="train")
texts = [preprocess(text) for text in data["text"]]
y = np.array(data["label"])
train_text, test_text, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Training sentences: {len(y_train)}; testing sentences: {len(y_test)}")

Training sentences: 12800; testing sentences: 3200


## Part 2.3 — Evaluation

True classes are represented by rows of the confusion matrix, and predicted
classes by columns. Precision, recall, and F1 were calculated per class and
macro-averaged. Undefined metrics were set to zero. The matrix and metrics
were checked against scikit-learn for every model.

In [4]:
# Part 2.3: NumPy confusion matrix and metrics.
def calculate_metrics(y_true, y_pred):
    # Rows = true classes; columns = predicted classes.
    matrix = np.zeros((6, 6), dtype=int)
    np.add.at(matrix, (y_true, y_pred), 1)
    correct = np.diag(matrix)
    precision = correct / np.maximum(matrix.sum(axis=0), 1)
    recall = correct / np.maximum(matrix.sum(axis=1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-15)
    return matrix, precision, recall, f1


def evaluate(name, y_true, y_pred):
    matrix, precision, recall, f1 = calculate_metrics(y_true, y_pred)
    labels = np.arange(6)
    np.testing.assert_array_equal(
        matrix, confusion_matrix(y_true, y_pred, labels=labels)
    )
    for values, reference in zip(
        [precision, recall, f1], [precision_score, recall_score, f1_score]
    ):
        np.testing.assert_allclose(values, reference(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        ))
        np.testing.assert_allclose(values.mean(), reference(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ))

    print(f"\n{name}\nConfusion matrix (class order: {', '.join(CLASSES)}):")
    print(matrix)
    print(f"{'Class':10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    for label, p, r, f in zip(CLASSES, precision, recall, f1):
        print(f"{label:10} {p:10.4f} {r:10.4f} {f:10.4f}")
    print(f"{'Macro':10} {precision.mean():10.4f} {recall.mean():10.4f} {f1.mean():10.4f}")
    print("All NumPy metrics match scikit-learn.")
    return f1.mean()

## Part 2.1 — Naïve Bayes

Class priors and word likelihoods were estimated with add-one smoothing.
The class with the highest sum of log prior and word log likelihoods was
selected for prediction.

In [5]:
# Part 2.1: multinomial Naive Bayes, using only NumPy and basic Python.
def train_nb(documents, y, vocabulary_size):
    prior = np.bincount(y, minlength=6) / len(y)
    counts = np.ones((6, vocabulary_size))  # Laplace smoothing: add one.
    for ids, label in zip(documents, y):
        np.add.at(counts[label], ids, 1)  # Repeated words count repeatedly.
    likelihood = counts / counts.sum(axis=1, keepdims=True)
    return np.log(prior), np.log(likelihood)


def predict_nb(documents, log_prior, log_likelihood):
    return np.array([
        np.argmax(log_prior + log_likelihood[:, ids].sum(axis=1))
        for ids in documents
    ])

In [6]:
train_words = [text.split() for text in train_text]
test_words = [text.split() for text in test_text]
word_vocab = make_vocabulary(train_words)
train_ids = encode(train_words, word_vocab)
test_ids = encode(test_words, word_vocab)
print(f"Word vocabulary: {len(word_vocab)}")
prior, likelihood = train_nb(train_ids, y_train, len(word_vocab))
nb_predictions = predict_nb(test_ids, prior, likelihood)
evaluate("NumPy Naive Bayes", y_test, nb_predictions)

# Identical vocabulary and tokenization make the NB comparison fair.

Word vocabulary: 13584

NumPy Naive Bayes
Confusion matrix (class order: sadness, joy, love, anger, fear, surprise):
[[ 870   55    0    7    1    0]
 [  47 1019    2    2    2    0]
 [  61  152   42    5    1    0]
 [ 109   93    1  223    6    0]
 [ 112   76    1   15  182    1]
 [  36   56    0    2   16    5]]
Class       Precision     Recall         F1
sadness        0.7045     0.9325     0.8026
joy            0.7023     0.9506     0.8078
love           0.9130     0.1609     0.2736
anger          0.8780     0.5162     0.6501
fear           0.8750     0.4703     0.6118
surprise       0.8333     0.0435     0.0826
Macro          0.8177     0.5123     0.5381
All NumPy metrics match scikit-learn.


np.float64(0.538087049519005)

### Scikit-learn comparison

`CountVectorizer` and `MultinomialNB` were used with the same vocabulary,
tokenization, and smoothing as the NumPy implementation.

In [7]:
vectorizer = CountVectorizer(
    vocabulary=word_vocab, tokenizer=str.split, token_pattern=None, lowercase=False
)
train_counts = vectorizer.fit_transform(train_text)
test_counts = vectorizer.transform(test_text)
nb = MultinomialNB(alpha=1.0).fit(train_counts, y_train)
sklearn_nb_predictions = nb.predict(test_counts)
evaluate("scikit-learn Naive Bayes", y_test, sklearn_nb_predictions)
np.testing.assert_array_equal(nb_predictions, sklearn_nb_predictions)
print("The two Naive Bayes models also make identical predictions.")


scikit-learn Naive Bayes
Confusion matrix (class order: sadness, joy, love, anger, fear, surprise):
[[ 870   55    0    7    1    0]
 [  47 1019    2    2    2    0]
 [  61  152   42    5    1    0]
 [ 109   93    1  223    6    0]
 [ 112   76    1   15  182    1]
 [  36   56    0    2   16    5]]
Class       Precision     Recall         F1
sadness        0.7045     0.9325     0.8026
joy            0.7023     0.9506     0.8078
love           0.9130     0.1609     0.2736
anger          0.8780     0.5162     0.6501
fear           0.8750     0.4703     0.6118
surprise       0.8333     0.0435     0.0826
Macro          0.8177     0.5123     0.5381
All NumPy metrics match scikit-learn.
The two Naive Bayes models also make identical predictions.


## Part 2.2 — Logistic Regression

Binary bigram features were used. Only the IDs of the bigrams present in each
sentence were stored, giving a sparse binary representation.

Local word order can be represented by bigrams, including phrases such as
“not happy”. However, frequency and longer context are not retained, many
rare features are created, and unseen bigrams cannot be represented.

A six-class softmax model was trained using cross-entropy and SGD, with a
learning rate of 0.01 for 20 epochs. The training set was shuffled every epoch.

In [8]:
# Part 2.2: six-class softmax logistic regression, trained with SGD.
def train_lr(documents, y, vocabulary_size, epochs=20, learning_rate=0.01):
    weights = np.zeros((vocabulary_size, 6))
    bias = np.zeros(6)
    rng = np.random.default_rng(42)
    for epoch in range(epochs):
        loss = 0.0
        for i in rng.permutation(len(y)):  # Shuffle every epoch.
            ids = documents[i]
            scores = weights[ids].sum(axis=0) + bias
            probabilities = np.exp(scores - scores.max())  # Stable softmax.
            probabilities /= probabilities.sum()
            loss -= np.log(max(probabilities[y[i]], 1e-15))
            gradient = probabilities.copy()
            gradient[y[i]] -= 1  # Cross-entropy gradient: p - one_hot(y).
            weights[ids] -= learning_rate * gradient
            bias -= learning_rate * gradient
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch + 1:2}: mean training loss = {loss / len(y):.4f}")
    return weights, bias


def predict_lr(documents, weights, bias):
    # Softmax preserves the ordering, so argmax(scores) gives the same label.
    return np.array([
        np.argmax(weights[ids].sum(axis=0) + bias) for ids in documents
    ])

In [9]:
train_bigrams = [get_bigrams(text) for text in train_text]
test_bigrams = [get_bigrams(text) for text in test_text]
bigram_vocab = make_vocabulary(train_bigrams)
train_ids = encode(train_bigrams, bigram_vocab, binary=True)
test_ids = encode(test_bigrams, bigram_vocab, binary=True)
print(f"\nBigram vocabulary: {len(bigram_vocab)}")
# [2, 7] means features 2 and 7 are 1; all other features are 0.
# Summing their weights is exactly x @ weights for this binary vector.
start = perf_counter()
weights, bias = train_lr(train_ids, y_train, len(bigram_vocab))
elapsed = perf_counter() - start
score = evaluate("NumPy Logistic Regression", y_test, predict_lr(test_ids, weights, bias))
results = [("NumPy Logistic Regression", score, elapsed)]


Bigram vocabulary: 91206


Epoch  5: mean training loss = 1.0040


Epoch 10: mean training loss = 0.6940


Epoch 15: mean training loss = 0.5263


Epoch 20: mean training loss = 0.4225

NumPy Logistic Regression
Confusion matrix (class order: sadness, joy, love, anger, fear, surprise):
[[687 211   1  20  14   0]
 [153 878  17  16   8   0]
 [ 59 147  43   9   2   1]
 [115 143   3 157  14   0]
 [102  98   3  13 166   5]
 [ 32  44   2   2  16  19]]
Class       Precision     Recall         F1
sadness        0.5984     0.7363     0.6603
joy            0.5773     0.8190     0.6772
love           0.6232     0.1648     0.2606
anger          0.7235     0.3634     0.4838
fear           0.7545     0.4289     0.5470
surprise       0.7600     0.1652     0.2714
Macro          0.6728     0.4463     0.4834
All NumPy metrics match scikit-learn.


### Scikit-learn comparison

The same binary bigram features were provided to `LogisticRegression` and
`SGDClassifier(loss="log_loss")`. Feature construction and evaluation were
excluded from the training times.

In [10]:
vectorizer = CountVectorizer(
    vocabulary=bigram_vocab, ngram_range=(2, 2), binary=True,
    tokenizer=str.split, token_pattern=None, lowercase=False
)
train_binary = vectorizer.fit_transform(train_text)
test_binary = vectorizer.transform(test_text)
for name, model in [
    ("sklearn LogisticRegression", LogisticRegression(max_iter=500)),
    ("sklearn SGDClassifier", SGDClassifier(loss="log_loss", random_state=42)),
]:
    start = perf_counter()
    model.fit(train_binary, y_train)
    elapsed = perf_counter() - start
    score = evaluate(name, y_test, model.predict(test_binary))
    results.append((name, score, elapsed))


sklearn LogisticRegression
Confusion matrix (class order: sadness, joy, love, anger, fear, surprise):
[[727 171   2  21  11   1]
 [158 874  16  16   7   1]
 [ 60 127  65   7   2   0]
 [111 116   4 187  14   0]
 [ 94  85   1  14 186   7]
 [ 32  39   2   3  16  23]]
Class       Precision     Recall         F1
sadness        0.6151     0.7792     0.6875
joy            0.6190     0.8153     0.7037
love           0.7222     0.2490     0.3704
anger          0.7540     0.4329     0.5500
fear           0.7881     0.4806     0.5971
surprise       0.7188     0.2000     0.3129
Macro          0.7029     0.4928     0.5369
All NumPy metrics match scikit-learn.

sklearn SGDClassifier
Confusion matrix (class order: sadness, joy, love, anger, fear, surprise):
[[758 156   2   9   7   1]
 [179 858  18  12   5   0]
 [ 66 125  62   6   2   0]
 [127 120   2 172  11   0]
 [106  81   3  11 179   7]
 [ 34  39   2   1  17  22]]
Class       Precision     Recall         F1
sadness        0.5969     0.8124     0.

### Results

In [11]:
print("\nLogistic regression comparison (same binary bigram features):")
for name, score, elapsed in results:
    print(f"{name:28} macro F1 = {score:.4f}; training = {elapsed:.2f}s")
print("Highest test macro F1 in this run:", max(results, key=lambda row: row[1])[0])


Logistic regression comparison (same binary bigram features):
NumPy Logistic Regression    macro F1 = 0.4834; training = 2.37s
sklearn LogisticRegression   macro F1 = 0.5369; training = 3.12s
sklearn SGDClassifier        macro F1 = 0.5283; training = 0.06s
Highest test macro F1 in this run: sklearn LogisticRegression


### Discussion

Identical predictions and a macro F1 of **0.5381** were obtained with both
Naïve Bayes implementations. Among the bigram logistic models, the highest
macro F1 was obtained with `LogisticRegression` (**0.5369**), followed by
`SGDClassifier` (**0.5283**) and the NumPy implementation (**0.4834**).
The shortest training time was recorded for `SGDClassifier`.

`LogisticRegression` can be used for moderate datasets when convergence is
practical. SGD can be used for large datasets and incremental training;
performance is affected by the learning rate and training duration.

The closest training approach is provided by `SGDClassifier`, since stochastic
updates are used in both implementations. Multiclass softmax was used in the
NumPy model and is also used by the default `LogisticRegression` solver.
One-vs-rest classification is used by `SGDClassifier`. L2 regularization is
enabled by default in both scikit-learn models and was omitted from the
NumPy implementation.

These results are limited to this split and these settings. Different features
were used for Naïve Bayes and logistic regression, so their score difference
cannot be attributed to the classifier alone.